# DANSUM FunctionGemma 웹용 변환
학습된 모델을 Transformers.js WebGPU에서 실행할 수 있는 ONNX q4f16 형식으로 변환해 공개 저장소에 업로드합니다. Colab 비밀 설정의 `HF_TOKEN`을 사용합니다.

In [ ]:
!pip -q install --upgrade 'optimum[onnxruntime]' onnxconverter-common huggingface_hub streamlit pyyaml
!git clone -q https://huggingface.co/spaces/onnx-community/convert-to-onnx /content/convert-to-onnx
print('변환 도구 준비 완료')

In [ ]:
import os, shutil, sys, time
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi, login

TOKEN = userdata.get('HF_TOKEN')
assert TOKEN, 'Colab 비밀 설정에서 HF_TOKEN의 노트북 액세스를 허용하세요.'
os.environ['HF_TOKEN'] = TOKEN
login(token=TOKEN, add_to_git_credential=False)
api = HfApi(token=TOKEN)
print('Hugging Face 연결:', api.whoami()['name'])

In [ ]:
sys.path.insert(0, '/content/convert-to-onnx')
from app import Config, ModelConverter
import onnx
from onnxconverter_common import float16 as onnx_float16
from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer

SOURCE = 'janyty/browsertools-functiongemma-270m'
TARGET = 'janyty/browsertools-functiongemma-270m-ONNX'
OUTPUT = Path('/content/dansum-functiongemma-onnx')
shutil.rmtree(OUTPUT, ignore_errors=True)

config = Config(hf_token=TOKEN, hf_username='janyty', is_using_user_token=True)
converter = ModelConverter(config)
started = time.perf_counter()
ok, log = converter._export_base(SOURCE, OUTPUT, extra_args=['--task', 'text-generation-with-past'])
print(log)
assert ok, 'ONNX 내보내기에 실패했습니다.'
print(f'기본 ONNX 변환: {time.perf_counter() - started:.1f}초')

In [ ]:
onnx_dir = OUTPUT / 'onnx'
base_files = [p for p in onnx_dir.glob('*.onnx') if not p.stem.endswith(('_q4', '_q4f16'))]
assert base_files, '변환된 ONNX 파일이 없습니다.'
quant_started = time.perf_counter()
for base_file in base_files:
    print('4비트 양자화:', base_file.name)
    model = onnx.load(str(base_file), load_external_data=True)
    quantizer = MatMulNBitsQuantizer(model, bits=4, block_size=32, is_symmetric=True)
    quantizer.process()
    q4_model = quantizer.model.model
    q4f16_model = onnx_float16.convert_float_to_float16(q4_model, keep_io_types=True)
    target = onnx_dir / f'{base_file.stem}_q4f16.onnx'
    onnx.save(q4f16_model, str(target))
    del model, quantizer, q4_model, q4f16_model

for path in list(onnx_dir.iterdir()):
    if path.is_file() and not path.name.endswith('_q4f16.onnx'):
        path.unlink()
print(f'q4f16 양자화: {time.perf_counter() - quant_started:.1f}초')
print('업로드 파일:', [p.name for p in onnx_dir.iterdir()])

In [ ]:
readme = OUTPUT / 'README.md'
readme.write_text('''---
library_name: transformers.js
pipeline_tag: text-generation
base_model:
- janyty/browsertools-functiongemma-270m
license: gemma
---

# DANSUM FunctionGemma 270M ONNX

Browser-ready q4f16 ONNX export of the fine-tuned DANSUM conversion router.
''', encoding='utf-8')
api.create_repo(TARGET, repo_type='model', private=False, exist_ok=True)
api.upload_folder(folder_path=str(OUTPUT), repo_id=TARGET, repo_type='model', commit_message='Add browser-ready q4f16 ONNX model')
print('완료: https://huggingface.co/' + TARGET)